# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Medical Image Formats (DICOM) and Segmentation Evaluation Metrics
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/labs/10_DICOM-metric.ipynb)

**Course**: Image and System Analysis

**Level**: Undergraduate / Graduate Computational Lab

**Target Audience**: Medical Physicists, Computational Researchers, Biomedical Engineers, Image and Signal Processing Students

**Author**: `Mehdi Astaraki`

---

## Overview & Learning Objectives
In the previous labs we treated images as plain 2D arrays of numbers. Real clinical images are more than that: every voxel lives at a **physical position inside the patient**, has a **physical size** in millimetres, and carries a **physical unit** (for CT, Hounsfield Units). This information is stored in the image header, and ignoring it leads to flipped left/right anatomy, distorted reformats and wrong distances.

Once an image has been segmented (Labs 08 and 09), the next question is: **how good is the segmentation?** The answer depends strongly on *how* we measure. Global voxel-counting metrics can hide missed small lesions, and a mask that is not what we expect (e.g. not binary) can silently produce wrong numbers.

This lab connects both topics into one clinical image-analysis workflow:
$$\underbrace{\text{DICOM series}}_{\text{scanner / PACS}} \;\longrightarrow\; \underbrace{\text{3D volume + affine}}_{\text{geometry, HU}} \;\longrightarrow\; \underbrace{\text{NIfTI}}_{\text{analysis format}} \;\longrightarrow\; \underbrace{\text{segmentation}}_{\text{algorithm}} \;\longrightarrow\; \underbrace{\text{evaluation}}_{\text{global + lesion-wise}}$$

By completing this notebook, you will master:
1. **DICOM Series Handling**: Why filenames cannot be trusted to give slice order, and how to read DICOM **tags** (by keyword, by tag number, safely with defaults) with `pydicom`.
2. **Interpreting CT Metadata**: The most important patient, acquisition, geometry and pixel tags of a CT image.
3. **Slice Geometry & Ordering**: Using `ImagePositionPatient` and `ImageOrientationPatient` to find the physical location and the correct order of slices.
4. **The Affine Matrix & Orientation**: Building the voxel-to-patient affine, working out the orientation (LPS, RAS, ...) and converting between DICOM and NIfTI conventions.
5. **Voxel Spacing & Intensities**: Pixel spacing vs. slice thickness vs. spacing between slices, and conversion of stored values into **Hounsfield Units (HU)**.
6. **3D Reconstruction & Display**: Axial, coronal and sagittal views with correct aspect ratio, CT windowing and the radiological display convention.
7. **From DICOM to NIfTI**: Reading the same series with **SimpleITK**, cross-validating our manual geometry, and exporting to NIfTI.
8. **Global Segmentation Metrics**: Dice, Precision, Recall and the 95th-percentile Hausdorff distance (HD95), implemented from first principles and unit-tested.
9. **Pitfalls of Naive Evaluation**: Multiple lesions merged into one object, and non-binary prediction masks that silently corrupt scores.
10. **Lesion-wise Evaluation**: Connected components, IoU-based **Hungarian lesion matching**, detection metrics and per-lesion metrics.

---

## Reference
The examined DICOM files in this lab were adopted from `CancerImagingArchive`:

**TCGA-LUSC | The Cancer Genome Atlas Lung Squamous Cell Carcinoma Collection TCIA**

`
Kirk, S., Lee, Y., Kumar, P., Filippini, J., Albertina, B., Watson, M., Rieger-Christ, K., & Lemmerman, J. (2016). The Cancer Genome Atlas Lung Squamous Cell Carcinoma Collection (TCGA-LUSC) (Version 4) [Data set]. The Cancer Imaging Archive. https://doi.org/10.7937/K9/TCIA.2016.TYGKKFMQ
`

---
# PART 0: Environment Setup, Data Assets, & Reproducibility

---

## SECTION 0.1: Environment Initialization

### Libraries Used in This Lab
Besides the familiar scientific-Python stack, this lab needs three libraries specialised for medical images:

| Library | Purpose |
|---|---|
| **`pydicom`** | Reading DICOM files: header (tags) and pixel data |
| **`SimpleITK`** | Reading/writing medical volumes (DICOM series, NIfTI) together with their geometry (spacing, origin, direction) |
| **`nibabel`** | Neuroimaging/NIfTI I/O; used here to cross-check orientation codes |
| `numpy`, `scipy`, `pandas`, `matplotlib`, `ipywidgets` | Arrays, connected components / distance transforms / Hungarian algorithm, tables, plots, interactive sliders |

The first cell installs any missing package automatically, so the notebook runs unchanged both on **Google Colab** and on a local Jupyter installation.

### Reproducibility
Some cells pick "random" slices to inspect. We fix the seed of Python's `random` module (`random.seed(42)`) so that every student sees the **same** slices and the interpretations in the text match the outputs. Change the seed (or delete the line) to explore other slices.

In [ ]:
# Cell 0.1.1: Automated Package Installation (Google Colab & Local)
import os
import sys
import importlib

# Map: import name -> pip package name
REQUIRED_PACKAGES = {
    "pydicom": "pydicom",
    "SimpleITK": "SimpleITK",
    "nibabel": "nibabel",
    "ipywidgets": "ipywidgets",
    "pandas": "pandas",
}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing missing package '{pip_name}'...")
        os.system(f"{sys.executable} -m pip install -q {pip_name}")

# On Google Colab, enable the widget manager so that ipywidgets sliders are interactive
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Running on Google Colab: custom widget manager enabled.")
except ImportError:
    print("Running outside Google Colab (local Jupyter).")

In [ ]:
# Cell 0.1.2: Environment Initialization & Replicability Configuration
# --- Standard library -------------------------------------------------------
import os        # file and folder operations (listing a directory, joining paths)
import random    # used to pick random slices
import tempfile  # temporary folder for files we write ourselves

# --- Third-party libraries ----------------------------------------------------
import numpy as np                 # arrays, linear algebra
import pandas as pd                # tables, for displaying tags and results nicely
import matplotlib
import matplotlib.pyplot as plt    # plotting
from matplotlib.colors import ListedColormap  # custom colours for the masks
import scipy as sp
from scipy import ndimage          # connected components, distance transforms
from scipy.optimize import linear_sum_assignment  # Hungarian algorithm (lesion matching)
import pydicom                     # reading DICOM files
from pydicom.datadict import tag_for_keyword, keyword_for_tag
import SimpleITK as sitk           # medical image I/O (keeps spacing, origin, orientation)
import nibabel as nib              # NIfTI I/O and orientation codes
import ipywidgets as widgets       # interactive sliders
from ipywidgets import interact    # connect a slider to a plotting function

# Fix the random seed so that every student gets the SAME "random" slices.
# Change the number (or delete this line) to explore other slices.
random.seed(42)
np.random.seed(0)

# Pandas display: 3 decimals, and do not truncate long texts
pd.set_option("display.precision", 3)
pd.set_option("display.max_colwidth", None)

# Configure inline plotting defaults
%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11

print("=== Environment Successfully Initialized ===")
print(f"Python Version:     {sys.version.split()[0]}")
print(f"NumPy Version:      {np.__version__}")
print(f"SciPy Version:      {sp.__version__}")
print(f"Pandas Version:     {pd.__version__}")
print(f"Matplotlib Version: {matplotlib.__version__}")
print(f"pydicom Version:    {pydicom.__version__}")
print(f"SimpleITK Version:  {sitk.Version_VersionString()}")
print(f"nibabel Version:    {nib.__version__}")

---
## SECTION 0.2: Data Assets

All data live in the folder `example_data/` next to this notebook. If a file is missing (e.g. on a fresh Google Colab runtime), it is downloaded automatically from the course GitHub repository.

| File | Used in | Content |
|---|---|---|
| `dicom/*.dcm` (10 files) | Part 1 | One CT series of 10 axial slices, anonymised (TCGA dataset from *The Cancer Imaging Archive*) |
| `ct.nii.gz` | Part 2 | A CT volume in NIfTI format (intensities in HU) |
| `reference.nii.gz` | Part 2 | The **reference** (ground-truth) lesion segmentation, drawn by an expert |
| `pred1.nii.gz`, `pred2.nii.gz`, `pred3.nii.gz` | Part 2 | Predictions of three segmentation algorithms |

> ⏳ `ct.nii.gz` is about 38 MB, so the first download on Colab can take a few seconds.

In [ ]:
# Cell 0.2.1: Automated Download & Verification of Data Assets
# Folders that contain the data (change these if your data lives somewhere else)
DATA_DIR = "./example_data"
DICOM_DIR = os.path.join(DATA_DIR, "dicom")
os.makedirs(DICOM_DIR, exist_ok=True)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Astarakee/isa-su/main/labs/example_data"

# The DICOM filenames are random UIDs, so they have to be listed explicitly
DICOM_FILES = [
    "0121cde6-b551-4e4e-82d9-b88a681f2d5a.dcm",
    "1ba9e2c8-05fe-419c-bd6a-acc9b0b0cdca.dcm",
    "215c9b84-50db-4b70-aed5-dd85850df507.dcm",
    "2ef7a575-9572-4418-aa06-d39b6167a577.dcm",
    "507357c9-0b83-4200-8209-9f809afd7990.dcm",
    "63b64dcf-ac80-44b1-9214-7be95dc6d545.dcm",
    "8495a20d-ebc2-4610-84bb-fc3b6887e85b.dcm",
    "9c980d55-96c8-4ff2-afeb-050ec446b8b1.dcm",
    "9f6005b2-7ba0-4e4e-874e-8670c030c651.dcm",
    "e6b2f03c-8b3d-42ac-bceb-804e3ecaafb8.dcm",
]
NIFTI_FILES = ["ct.nii.gz", "reference.nii.gz", "pred1.nii.gz", "pred2.nii.gz", "pred3.nii.gz"]

ASSETS = {f"dicom/{f}": f"{GITHUB_RAW_BASE}/dicom/{f}" for f in DICOM_FILES}
ASSETS.update({f: f"{GITHUB_RAW_BASE}/{f}" for f in NIFTI_FILES})

import urllib.request

for rel_path, url in ASSETS.items():
    fpath = os.path.join(DATA_DIR, rel_path)
    if not os.path.exists(fpath):
        print(f"Downloading missing asset '{rel_path}'...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as resp, open(fpath, 'wb') as f:
                f.write(resp.read())
        except Exception as e:
            print(f"Warning: Failed to download '{rel_path}': {e}")

missing = [p for p in ASSETS if not os.path.exists(os.path.join(DATA_DIR, p))]
assert not missing, f"Missing data files: {missing}"
print(f"All {len(ASSETS)} data assets are available in '{DATA_DIR}'.")

---
# PART 1: Loading and Managing DICOM Files

---

### Learning objectives of Part 1
By the end of this part you should be able to:

1. Load a DICOM series from a folder and explain why **filenames cannot be trusted** to give slice order.
2. Read DICOM **tags (metadata)** in several different ways.
3. Interpret the most important tags of a CT image.
4. Use geometric tags to find the **physical location** and the **order** of slices.
5. Build the **affine matrix** (voxel index → patient coordinates) and work out the image **orientation** (LPS, RAS, ...).
6. Extract the **voxel spacing**: pixel spacing, slice thickness and spacing between slices.
7. Convert stored pixel values into **Hounsfield Units (HU)** and find the intensity range.
8. Stack the slices into a 3D volume and display **axial, coronal and sagittal** views.

---

### What is DICOM?

**DICOM** (*Digital Imaging and Communications in Medicine*) is the international standard for storing and sending medical images. Each DICOM file (`.dcm`) holds two things:

| Part | Contents |
|---|---|
| **Header (metadata)** | A list of *data elements* (also called **tags**): patient information, scanner settings, image geometry, ... |
| **Pixel data** | The image itself, stored in tag `(7FE0,0010)` |

For CT and MRI, a 3D scan is usually stored as a **series** of 2D files, **one file per slice**. Rebuilding the 3D volume correctly is our job, and the header contains everything needed to do it.

We will use the **`pydicom`** library to read the files, **`numpy`** for the arrays, and **`matplotlib`** for the plots.

---
## SECTION 1: Listing the DICOM Files — Why Filenames Are Not Enough

First we list the files in the folder. A beginner might think: *"I'll sort the files alphabetically and that gives me slice 1, 2, 3, ..."*. Let's see if that works.

In [ ]:
# Cell 1.1: Listing the Files of the Series
# os.listdir() returns the names of all entries in the folder.
# We keep only the files ending in ".dcm" and sort them alphabetically.
filenames = sorted(f for f in os.listdir(DICOM_DIR) if f.lower().endswith(".dcm"))

print(f"Found {len(filenames)} DICOM files in '{DICOM_DIR}':\n")
for i, name in enumerate(filenames):
    # i is just the position in the ALPHABETICAL list, nothing more
    print(f"  alphabetical #{i:2d}  ->  {name}")

### Discussion: can we sort the slices from these filenames?

**No.** The filenames here are random **UUIDs** (*Universally Unique Identifiers*), e.g. `0121cde6-b551-...dcm`. They are unique, but they say **nothing** about where a slice sits in the body. Sorting them alphabetically gives an arbitrary order.

This is common in practice:

* **PACS exports and anonymisation tools** often rename files to UIDs or hashes to remove patient information.
* Even names that *look* ordered (`IM0001`, `IM0002`, ...) can be misleading: they may reflect **export order** rather than anatomical order, numbering may restart for each series, and plain alphabetical sorting puts `IM10` before `IM2`.
* A single folder can contain **several series** mixed together (e.g. a scout image, a contrast scan, a non-contrast scan).

**Lesson:** the slice order must come from the **DICOM header** (the tags), never from the filename. Sections 2 to 4 show how.

In [ ]:
# Cell 1.2: Instance Numbers in Alphabetical File Order
# A quick demonstration: read the "Instance Number" tag (the slice number the
# scanner assigned) of each file, keeping the alphabetical order.
# We will look at tags in detail in the next section.
for name in filenames:
    # stop_before_pixels=True reads ONLY the header. This is much faster
    # when we do not need the image itself.
    ds = pydicom.dcmread(os.path.join(DICOM_DIR, name), stop_before_pixels=True)
    print(f"{name}  ->  InstanceNumber = {ds.InstanceNumber}")

# Notice that the instance numbers come out jumbled (3, 9, 6, 7, ...):
# alphabetical order != slice order.

---
## SECTION 2: How DICOM Tags Work and How to Fetch Them

Each piece of metadata is a **data element** with:

| Field | Example | Meaning |
|---|---|---|
| **Tag** | `(0010,0010)` | Two hexadecimal numbers: *(group, element)*. The group roughly gives the category, e.g. `0010` = patient, `0018` = acquisition, `0020` = geometry/relationships, `0028` = image pixel description |
| **Keyword** | `PatientName` | Human-readable name defined by the DICOM dictionary |
| **VR** (Value Representation) | `PN` | Data type: `PN` = person name, `DS` = decimal string, `US` = unsigned short, `UI` = UID, `CS` = code string, ... |
| **Value** | `TCGA-34-5231` | The actual content |

Tags with an **odd group number** (e.g. `0009`, `0019`, `0043`) are **private tags**. Each manufacturer (here GE) defines its own, and the meaning is not standardised.

With `pydicom` there are several ways to fetch a tag:

1. **By keyword (attribute access):** `ds.PatientName`. Easiest to read.
2. **By tag number:** `ds[0x0010, 0x0010]` returns the full *DataElement* (tag, VR, value); add `.value` to get the value only.
3. **With `ds.get(keyword, default)`:** like a Python dictionary. It returns `default` instead of raising an error when the tag is missing. This is the safe choice, because **not every file contains every tag**.
4. **Membership test:** `"SliceLocation" in ds` tells you if a tag exists.
5. **Iteration:** `for elem in ds:` walks over all elements.

Let's try all of them on the first file.

In [ ]:
# Cell 2.1: Five Ways to Access a Tag
# Read one file completely (header + pixel data)
example_path = os.path.join(DICOM_DIR, filenames[0])
ds = pydicom.dcmread(example_path)

# --- 1. By keyword ----------------------------------------------------------
print("1) Keyword access      : ds.Modality              ->", ds.Modality)

# --- 2. By (group, element) tag number ---------------------------------------
elem = ds[0x0008, 0x0060]          # returns a DataElement object
print("2) Tag-number access   : ds[0x0008, 0x0060]       ->", elem)
print("                         ds[0x0008, 0x0060].value ->", elem.value)
print("                         VR (data type)           ->", elem.VR)
print("                         keyword                  ->", elem.keyword)

# --- 3. Safe access with a default value ------------------------------------
# 'SpacingBetweenSlices' (0018,0088) is optional and is NOT present in this file.
print("3) Safe access         : ds.get('SpacingBetweenSlices', 'NOT PRESENT') ->",
      ds.get("SpacingBetweenSlices", "NOT PRESENT"))

# --- 4. Membership test -----------------------------------------------------
print("4) Membership test     : 'SliceLocation' in ds    ->", "SliceLocation" in ds)

# --- Converting between keywords and tag numbers ----------------------------
# (tag_for_keyword and keyword_for_tag were imported in Cell 0.1.2)
print("\nKeyword -> tag number  : PatientID ->", hex(tag_for_keyword("PatientID")))
print("Tag number -> keyword  : 0x00280030 ->", keyword_for_tag(0x00280030))

In [ ]:
# Cell 2.2: Iterating Over Data Elements
# --- 5. Iterate over elements -------------------------------------------------
# Printing the whole dataset (print(ds)) is very long, so we show only the
# first 15 elements to see what the header looks like.
for i, elem in enumerate(ds):
    if i >= 15:
        break
    print(elem)

print("...")
print(f"\nThis file contains {len(ds)} top-level data elements in total.")

In [ ]:
# Cell 2.3: Private (Vendor-Specific) Tags
# Count the private (manufacturer-specific) tags: their group number is ODD.
private = [elem for elem in ds if elem.tag.is_private]
print(f"Number of private tags: {len(private)}")
print("A few examples (meaning defined by the vendor, here GE):")
for elem in private[:5]:
    print("  ", elem)

> **Tip.** `print(ds)` shows the full header. Try it in a new cell. Pixel data are shown only as a short summary, so it is safe to print.

---
## SECTION 3: Exploring 20 Important Tags of a Random Slice

Now we pick **one random slice** and look at 20 commonly used tags. We store them in a table that shows the tag number, keyword, value, and a short explanation.

In [ ]:
# Cell 3.1: Table of 20 Important CT Tags
# Pick one random filename from the list
random_file = random.choice(filenames)
print("Randomly selected file:", random_file)

ds_random = pydicom.dcmread(os.path.join(DICOM_DIR, random_file))

# The 20 tags we want to inspect, each with a short explanation.
# They are grouped by category: patient/study, acquisition, geometry, pixels.
tags_of_interest = {
    # ----- Patient / study / series identification ----------------------------
    "PatientID":               "Patient identifier (anonymised here)",
    "PatientSex":              "Patient's sex (M/F/O)",
    "PatientAge":              "Age at the time of the study (072Y = 72 years)",
    "StudyDate":               "Date the study was done (YYYYMMDD; dates may be shifted by anonymisation)",
    "Modality":                "Type of equipment: CT, MR, PT, US, ...",
    "StudyDescription":        "Free-text description of the study",
    "SeriesDescription":       "Free-text description of this series",
    "BodyPartExamined":        "Anatomical region scanned",
    # ----- Acquisition parameters ----------------------------------------------
    "Manufacturer":            "Scanner manufacturer",
    "KVP":                     "X-ray tube peak voltage (kV)",
    "ConvolutionKernel":       "Reconstruction filter: BONE/LUNG = sharp, STANDARD/SOFT = smooth",
    "PatientPosition":         "Patient position on the table, e.g. HFS = Head First Supine, FFP = Feet First Prone",
    # ----- Geometry (where is the slice in space?) ----------------------------
    "InstanceNumber":          "Image number assigned by the scanner (often, but not always, the slice order)",
    "ImagePositionPatient":    "x,y,z (mm) of the CENTRE of the first (top-left) voxel in patient coordinates",
    "ImageOrientationPatient": "Direction cosines of the image rows and columns in patient coordinates",
    "SliceThickness":          "Nominal thickness of the reconstructed slice (mm)",
    "PixelSpacing":            "Physical distance between pixel centres: [row spacing, column spacing] (mm)",
    # ----- Pixel data description ---------------------------------------------
    "Rows":                    "Number of pixel rows (image height)",
    "Columns":                 "Number of pixel columns (image width)",
    "RescaleIntercept":        "b in: real value = slope * stored value + b (gives Hounsfield Units for CT)",
}

# Build a table row for each tag
rows = []
for keyword, description in tags_of_interest.items():
    tag = tag_for_keyword(keyword)            # e.g. 0x00100020
    value = ds_random.get(keyword, "N/A")     # safe access
    rows.append({
        "Tag":         f"({tag >> 16:04X},{tag & 0xFFFF:04X})",  # format as (gggg,eeee)
        "Keyword":     keyword,
        "Value":       str(value),
        "Description": description,
    })

tags_table = pd.DataFrame(rows)
tags_table

### Interpretation of the tags

* **Patient/study tags.** This is an anonymised chest CT (`TCGA-...` is a dataset from *The Cancer Imaging Archive*) of a 72-year-old male. The dates can be shifted on purpose during anonymisation.
* **`Modality = CT`.** This tells us the pixel values can be converted into **Hounsfield Units (HU)**, where air ≈ −1000, water = 0 and bone ≈ +400 to +1000.
* **`ConvolutionKernel = BONE`.** This is a *sharp* reconstruction kernel. Edges look crisp (good for lung and bone detail) but the image is noisier.
* **`PatientPosition = FFP`.** *Feet First Prone*: the patient lay face down. **Remember this**, because it explains the unusual orientation we will find in Section 5.
* **`ImagePositionPatient` (IPP)** and **`ImageOrientationPatient` (IOP)** are the two **most important geometric tags**. Together with `PixelSpacing` they place every voxel in 3D space (Sections 4 and 5).
* **`SliceThickness`** is the thickness of each slice. It is *not necessarily* the distance between slices (Section 6).
* **`Rows` × `Columns` = 512 × 512**, the usual CT matrix size.
* **`RescaleIntercept`** (together with `RescaleSlope`) converts the **stored integers** into real physical values (Section 7).

---
## SECTION 4: Finding Where Two Random Slices Are Located in the Stack

Which tags tell us the **position** of a slice? There are three candidates:

| Tag | Reliable? | Comment |
|---|---|---|
| `InstanceNumber (0020,0013)` | ⚠️ Usually | A counter set by the scanner. It can start at any value, count in either direction, or have gaps |
| `SliceLocation (0020,1041)` | ⚠️ Not always | Optional and vendor-dependent. Meaningless for oblique slices |
| `ImagePositionPatient (0020,0032)` + `ImageOrientationPatient (0020,0037)` | ✅ **Yes** | The standard, geometric way |

### The geometric method

`ImageOrientationPatient` stores 6 numbers: the **row direction** $\mathbf{r}$ (3 numbers) and the **column direction** $\mathbf{c}$ (3 numbers). Their **cross product** gives the **slice normal** $\mathbf{n}$, the direction perpendicular to the slices:

$$\mathbf{n} = \mathbf{r} \times \mathbf{c}$$

Projecting the position of each slice (IPP) onto the normal gives one number per slice, its **distance along the stacking axis**:

$$d = \mathbf{n} \cdot \mathbf{IPP}$$

Sorting the slices by $d$ gives the correct anatomical order. This works for **any** orientation (axial, sagittal, coronal or oblique).

In [ ]:
# Cell 4.1: Geometric Tags of Two Random Slices
# Pick two DIFFERENT random files
file_a, file_b = random.sample(filenames, 2)
print("Random slice A:", file_a)
print("Random slice B:", file_b)

# Read only their headers (pixels not needed here)
ds_a = pydicom.dcmread(os.path.join(DICOM_DIR, file_a), stop_before_pixels=True)
ds_b = pydicom.dcmread(os.path.join(DICOM_DIR, file_b), stop_before_pixels=True)

for label, d in [("A", ds_a), ("B", ds_b)]:
    print(f"\n--- Slice {label} ---")
    print("  InstanceNumber          :", d.InstanceNumber)
    print("  SliceLocation           :", d.get("SliceLocation", "N/A"))
    print("  ImagePositionPatient    :", list(d.ImagePositionPatient))
    print("  ImageOrientationPatient :", list(d.ImageOrientationPatient))

In [ ]:
# Cell 4.2: Position Along the Slice Normal
def slice_position_along_normal(d):
    # Return the slice normal and the position of the slice along it (mm).
    iop = np.array(d.ImageOrientationPatient, dtype=float)   # 6 numbers
    row_dir = iop[:3]    # direction in which the column index increases (along a row)
    col_dir = iop[3:]    # direction in which the row index increases (down a column)
    normal = np.cross(row_dir, col_dir)                      # perpendicular to the slice
    ipp = np.array(d.ImagePositionPatient, dtype=float)      # position of first voxel
    return normal, float(np.dot(normal, ipp))                # projection onto the normal

normal_a, pos_a = slice_position_along_normal(ds_a)
normal_b, pos_b = slice_position_along_normal(ds_b)

print("Slice normal n            :", normal_a)
print(f"Position of slice A along n: {pos_a:.3f} mm")
print(f"Position of slice B along n: {pos_b:.3f} mm")
print(f"Distance between A and B   : {abs(pos_a - pos_b):.3f} mm")

The two positions alone tell us which slice comes **first**, but not its **rank** in the whole stack (e.g. "slice 1" vs "slice 10"). For that we compute the position of **every** slice, sort them, and look up where A and B land.

In [ ]:
# Cell 4.3: Geometric Ranking of All Slices
# Compute the position along the normal for ALL slices
records = []
for name in filenames:
    d = pydicom.dcmread(os.path.join(DICOM_DIR, name), stop_before_pixels=True)
    _, pos = slice_position_along_normal(d)
    records.append({"filename": name,
                    "position_mm": pos,
                    "InstanceNumber": int(d.InstanceNumber),
                    "SliceLocation": float(d.get("SliceLocation", np.nan))})

# Sort by geometric position (ascending = moving along +n)
slices_df = pd.DataFrame(records).sort_values("position_mm").reset_index(drop=True)
slices_df.index = slices_df.index + 1          # human-friendly ranks: 1..N
slices_df.index.name = "geometric_rank"

# Find the rank of our two random slices
rank_a = slices_df.index[slices_df.filename == file_a][0]
rank_b = slices_df.index[slices_df.filename == file_b][0]
print(f"Slice A ({file_a}) is slice #{rank_a} of {len(slices_df)}")
print(f"Slice B ({file_b}) is slice #{rank_b} of {len(slices_df)}")

slices_df

### Discussion

* Sorting by the **geometric position** gives slices in steps of exactly 0.625 mm.
* Here `InstanceNumber` and `SliceLocation` happen to **agree** with the geometric order, which is common for CT scanners. Do not rely on that: in other datasets instance numbers can run in the opposite direction or have gaps, and `SliceLocation` can be missing.
* The normal is $\mathbf{n} = (0, 0, 1)$: the slices are stacked along the patient's **z-axis** (feet → head). In other words, these are **axial** slices, and a larger position means further toward the **head** (superior).

---
## SECTION 5: The Affine Matrix and the Image Orientation (LPS? RAS? ...)

### The DICOM patient coordinate system: LPS

DICOM defines the **patient coordinate system** (in mm) relative to the **patient's body**, not the scanner:

* **+x** points toward the patient's **L**eft
* **+y** points toward the patient's **P**osterior (back)
* **+z** points toward the patient's **S**uperior (head)

This is called **LPS**. (NIfTI and many neuroimaging tools such as FSL, FreeSurfer and nibabel use **RAS** instead: +x = Right, +y = Anterior, +z = Superior. Converting between them only flips the signs of x and y.)

### The affine matrix

The **affine** maps a voxel index $(i, j, k)$ (column, row, slice) to a position $(x, y, z)$ in mm:

$$
\begin{bmatrix} x \\ y \\ z \\ 1 \end{bmatrix} =
\underbrace{\begin{bmatrix}
r_x \Delta_c & c_x \Delta_r & n_x \Delta_s & T_x \\
r_y \Delta_c & c_y \Delta_r & n_y \Delta_s & T_y \\
r_z \Delta_c & c_z \Delta_r & n_z \Delta_s & T_z \\
0 & 0 & 0 & 1
\end{bmatrix}}_{\text{affine } A}
\begin{bmatrix} i \\ j \\ k \\ 1 \end{bmatrix}
$$

where
* $\mathbf{r}$ = row direction cosine (first 3 values of IOP), the direction of increasing column index $i$
* $\mathbf{c}$ = column direction cosine (last 3 values of IOP), the direction of increasing row index $j$
* $\mathbf{n}$ = slice normal $\mathbf{r} \times \mathbf{c}$, the direction of increasing slice index $k$
* $\Delta_c$ = `PixelSpacing[1]` (spacing between columns), $\Delta_r$ = `PixelSpacing[0]` (spacing between rows), $\Delta_s$ = spacing between slices
* $\mathbf{T}$ = `ImagePositionPatient` of the **first** slice (k = 0)

> ⚠️ **Classic pitfall:** `PixelSpacing` is stored as **[row spacing, column spacing]**, i.e. [Δ along j, Δ along i]. The order is the reverse of (x, y). It makes no difference when pixels are square, as here, but it matters for non-square pixels.

In [ ]:
# Cell 5.1: Assembling the 4x4 Affine Matrix
# Use the geometrically sorted list from Section 4
sorted_files = slices_df["filename"].tolist()

first = pydicom.dcmread(os.path.join(DICOM_DIR, sorted_files[0]),  stop_before_pixels=True)
last  = pydicom.dcmread(os.path.join(DICOM_DIR, sorted_files[-1]), stop_before_pixels=True)

# Direction cosines
iop = np.array(first.ImageOrientationPatient, dtype=float)
row_dir = iop[:3]                         # r : direction of increasing column index i
col_dir = iop[3:]                         # c : direction of increasing row index j
slice_dir = np.cross(row_dir, col_dir)    # n : direction of increasing slice index k

# Spacings
row_spacing, col_spacing = (float(v) for v in first.PixelSpacing)   # [Δr, Δc]
n_slices = len(sorted_files)
# Average slice step = total distance from first to last slice / (N - 1)
ipp_first = np.array(first.ImagePositionPatient, dtype=float)
ipp_last  = np.array(last.ImagePositionPatient,  dtype=float)
slice_spacing = np.dot(ipp_last - ipp_first, slice_dir) / (n_slices - 1)

# Assemble the 4x4 affine (LPS world coordinates)
affine_lps = np.eye(4)
affine_lps[:3, 0] = row_dir   * col_spacing     # column i -> moves along r
affine_lps[:3, 1] = col_dir   * row_spacing     # row    j -> moves along c
affine_lps[:3, 2] = slice_dir * slice_spacing   # slice  k -> moves along n
affine_lps[:3, 3] = ipp_first                   # translation = first voxel position

np.set_printoptions(precision=4, suppress=True)
print("Row direction r    :", row_dir)
print("Column direction c :", col_dir)
print("Slice normal n     :", slice_dir)
print("\nAffine matrix (voxel index -> LPS mm):\n", affine_lps)

In [ ]:
# Cell 5.2: Sanity Check of the Affine
# Sanity check: map the voxel (i=0, j=0, k=last) through the affine.
# The result should equal the ImagePositionPatient of the LAST slice.
predicted = affine_lps @ np.array([0, 0, n_slices - 1, 1])
print("Affine prediction for last slice origin:", predicted[:3])
print("IPP stored in the last slice           :", ipp_last)
print("Match?", np.allclose(predicted[:3], ipp_last, atol=1e-3))

### From the affine to orientation letters

To describe the orientation of the **voxel array**, we ask for each voxel axis: *"when this index increases, toward which side of the patient do we move?"* We take the dominant component of each column of the affine and turn it into a letter:

| Dominant component | Positive sign | Negative sign |
|---|---|---|
| x | L | R |
| y | P | A |
| z | S | I |

> ⚠️ **Warning about naming conventions.** Orientation strings are ambiguous across software. Here (as in **nibabel** and **3D Slicer**) a letter means *the direction the axis points **toward***. Some tools, notably **ITK/SimpleITK** "orientation codes", name the side the axis comes *from*, so the same image can be called "RAS" in one tool and "LPI" in another! Always check which convention a tool uses.

In [ ]:
# Cell 5.3: Orientation Codes from the Affine
def axis_codes(affine, world="LPS"):
    # Return, for each voxel axis (i, j, k), the patient direction it points TOWARD.
    # world = "LPS" means the affine's x/y/z are DICOM LPS coordinates;
    # world = "RAS" means NIfTI-style RAS coordinates.
    positive = {"LPS": "LPS", "RAS": "RAS"}[world]          # letter for + direction
    negative = {"LPS": "RAI", "RAS": "LPI"}[world]          # letter for - direction
    codes = []
    for axis in range(3):
        column = affine[:3, axis]                 # world direction of this voxel axis
        dominant = int(np.argmax(np.abs(column))) # which of x, y, z dominates
        sign = column[dominant]
        codes.append(positive[dominant] if sign > 0 else negative[dominant])
    return "".join(codes)

print("Voxel axis orientation (i, j, k):", axis_codes(affine_lps, world="LPS"))

### Interpretation: this image is **not** LPS-ordered!

Let's read the direction cosines:

* $\mathbf{r} = (-1, 0, 0)$: moving along a row (increasing column $i$) goes in **−x**, i.e. toward the patient's **Right**.
* $\mathbf{c} = (0, -1, 0)$: moving down a column (increasing row $j$) goes in **−y**, i.e. toward the patient's **Anterior**.
* $\mathbf{n} = (0, 0, +1)$: increasing slice index $k$ goes toward **Superior**.

So the voxel array is ordered **R-A-S** (i → Right, j → Anterior, k → Superior).

**Why?** Remember `PatientPosition = FFP` (*Feet First Prone*). The patient lies **face down**, so the patient's back faces the top of the scanner image and the left/right sides swap compared with a supine scan. For a normal **supine** (HFS) scan you would typically see IOP = `[1,0,0, 0,1,0]`, which gives the familiar **L-P-S** voxel order.

**Key point:** the **world coordinate system** of DICOM is **always LPS**. What changes from scan to scan is how the **voxel array** is laid out in that system, and that is exactly what the affine records. This is why we never assume orientation and always build it from IOP/IPP.

In [ ]:
# Cell 5.4: Converting the Affine to the NIfTI / RAS Convention
# Converting the affine to the NIfTI / RAS world convention:
# multiply by diag(-1, -1, 1, 1) to flip the x (L->R) and y (P->A) axes.
LPS_TO_RAS = np.diag([-1, -1, 1, 1])
affine_ras = LPS_TO_RAS @ affine_lps
print("Affine in RAS world coordinates (what a NIfTI file would store):\n", affine_ras)
print("\nVoxel axis orientation computed in RAS world:", axis_codes(affine_ras, world="RAS"))
print("(Same answer: changing the world convention does not change where the axes point.)")

# Cross-check with nibabel: it should agree with our function.
print("nibabel.aff2axcodes :", "".join(nib.aff2axcodes(affine_ras)))

---
## SECTION 6: Voxel Spacing — Pixel Spacing, Slice Thickness and Spacing Between Slices

A voxel has three physical dimensions:

| Quantity | Where it comes from | Meaning |
|---|---|---|
| **Pixel spacing** (in-plane) | `PixelSpacing (0028,0030)` | Distance between neighbouring pixel centres within a slice |
| **Slice thickness** | `SliceThickness (0018,0050)` | How "thick" the slab of tissue is that each slice averages over |
| **Spacing between slices** | Computed from **IPP** differences (or the optional tag `SpacingBetweenSlices (0018,0088)`) | Distance between the centres of adjacent slices |

**Slice thickness ≠ spacing between slices** in general:

* spacing **<** thickness → slices **overlap** (common in CT, to make smoother 3D reconstructions)
* spacing **>** thickness → there are **gaps** between slices (common in MRI)

For 3D analysis (resampling, measuring volumes, registration) the **spacing between slices** is what counts, **not** the slice thickness.

In [ ]:
# Cell 6.1: Extracting the Three Spacings
# In-plane pixel spacing (identical for every slice of a series)
print(f"PixelSpacing           : row spacing = {row_spacing} mm, column spacing = {col_spacing} mm")

# Slice thickness tag
print(f"SliceThickness (tag)   : {float(first.SliceThickness)} mm")

# Optional SpacingBetweenSlices tag; often absent in CT
print(f"SpacingBetweenSlices   : {first.get('SpacingBetweenSlices', 'tag not present in this file')}")

# Robust method: differences between consecutive slice positions along the normal
positions = slices_df["position_mm"].to_numpy()
gaps = np.diff(positions)
print(f"\nPositions along normal : {positions}")
print(f"Consecutive differences: {gaps}")
print(f"Spacing between slices : {gaps.mean():.4f} mm  (min {gaps.min():.4f}, max {gaps.max():.4f})")

# Check for uniform spacing. Missing slices would show up as larger gaps.
if np.allclose(gaps, gaps[0], atol=1e-3):
    print("✅ Slices are uniformly spaced: no missing slices.")
else:
    print("⚠️ Non-uniform spacing: some slices may be missing.")

In [ ]:
# Cell 6.2: Voxel Size and Slice Overlap
# Gather everything into the final voxel size (x = columns, y = rows, z = slices)
voxel_size = (col_spacing, row_spacing, gaps.mean())
print(f"Voxel size (i, j, k)   : {voxel_size[0]:.4f} x {voxel_size[1]:.4f} x {voxel_size[2]:.4f} mm")
print(f"Voxel volume           : {np.prod(voxel_size):.4f} mm^3")

thickness = float(first.SliceThickness)
overlap = thickness - gaps.mean()
print(f"\nSlice thickness ({thickness} mm) > spacing ({gaps.mean():.3f} mm) "
      f"-> neighbouring slices OVERLAP by {overlap:.3f} mm ({100 * overlap / thickness:.0f} %).")

### Interpretation

* In-plane resolution is **0.703 mm**. This equals the reconstruction field of view (360 mm, tag `ReconstructionDiameter`) divided by 512 pixels.
* Each slice is **1.25 mm thick**, but slices are spaced only **0.625 mm** apart: consecutive slices overlap by **50 %**.
* The voxels are **anisotropic** (0.703 × 0.703 × 0.625 mm). We must use this when displaying reformatted (coronal/sagittal) views, otherwise they will look stretched or squashed (Section 8).

---
## SECTION 7: Intensity Range — Stored Values vs. Hounsfield Units

The pixel data are stored as **integers** (here 16-bit signed, see `BitsAllocated` and `PixelRepresentation`). To get physically meaningful values we apply the **rescale** transform from the header:

$$\text{HU} = \text{RescaleSlope} \times \text{stored value} + \text{RescaleIntercept}$$

For CT the result is in **Hounsfield Units**:

| Tissue | Typical HU |
|---|---|
| Air | −1000 |
| Lung | −900 to −500 |
| Fat | −100 to −50 |
| Water | 0 |
| Soft tissue | +20 to +80 |
| Bone | +400 to +2000 |

> Slope and intercept can differ **from slice to slice**, so apply them **per slice**, before stacking.

In [ ]:
# Cell 7.1: Pixel Storage Tags
# Look at the tags that describe how pixels are stored
print("BitsAllocated       :", first.BitsAllocated)        # bits per pixel in the file
print("BitsStored          :", first.BitsStored)           # bits actually used
print("PixelRepresentation :", first.PixelRepresentation,  "(0 = unsigned, 1 = signed)")
print("RescaleSlope        :", first.RescaleSlope)
print("RescaleIntercept    :", first.RescaleIntercept)
print("PixelPaddingValue   :", first.get("PixelPaddingValue", "N/A"),
      "(value used to mark pixels outside the scanned region, if any)")

In [ ]:
# Cell 7.2: Converting Every Slice to HU
# Read every slice IN GEOMETRIC ORDER, convert to HU, and record min/max values
hu_slices = []
stats = []
for rank, name in enumerate(sorted_files, start=1):
    d = pydicom.dcmread(os.path.join(DICOM_DIR, name))     # header + pixels
    raw = d.pixel_array                                     # stored integers (numpy array)
    slope = float(d.RescaleSlope)
    intercept = float(d.RescaleIntercept)
    hu = raw.astype(np.float32) * slope + intercept         # -> Hounsfield Units
    hu_slices.append(hu)
    stats.append({"rank": rank, "raw_min": int(raw.min()), "raw_max": int(raw.max()),
                  "HU_min": hu.min(), "HU_max": hu.max()})

stats_df = pd.DataFrame(stats).set_index("rank")
stats_df

In [ ]:
# Cell 7.3: Global Intensity Range and HU Histogram
# Global minimum and maximum over the whole series
all_hu = np.stack(hu_slices)                  # temporary 3D array, used for the statistics
print(f"Stored (raw) values : min = {stats_df.raw_min.min()},  max = {stats_df.raw_max.max()}")
print(f"Hounsfield Units    : min = {all_hu.min():.0f} HU,  max = {all_hu.max():.0f} HU")

# Histogram of HU values: a picture of what tissues are present
plt.figure(figsize=(8, 3.5))
# Bins 20 HU wide, covering the full range of values in the volume
bin_edges = np.arange(all_hu.min(), all_hu.max() + 20, 20)
plt.hist(all_hu.ravel(), bins=bin_edges, color="steelblue", log=True)  # log scale shows small peaks
plt.xlabel("Hounsfield Units")
plt.ylabel("Voxel count (log scale)")
plt.title("Intensity histogram of the whole series")
for hu_value, label in [(-1000, "air"), (0, "water"), (1000, "bone")]:
    plt.axvline(hu_value, color="red", ls="--", lw=0.8)
    plt.text(hu_value, plt.ylim()[1] * 0.3, label, color="red", ha="center")
plt.tight_layout()
plt.show()

### Interpretation

* The **minimum** is about **−2048 HU**, which is physically impossible (nothing is less dense than air at −1000 HU). These values are the corners **outside the circular reconstruction field of view**, which the scanner fills with a fixed low value. They are not tissue, so mask or clip them before computing statistics (e.g. `np.clip(hu, -1024, None)`).
* The spike near **−1000 HU** is **air**, mostly outside the body plus some gas in the bowel.
* The broad peak around **0 to +60 HU** is **soft tissue** (liver, spleen, kidneys, muscle), with **fat** forming the shoulder around −100 HU.
* The tail above **+400 HU** is **bone** (spine, ribs).
* There is little lung tissue (−900 to −500 HU) because, despite the study name "CT HI RES CHEST", these 10 slices lie about 36 cm below the sternal notch (`PositionReferenceIndicator = SN`, z ≈ −359 mm), at the level of the **upper abdomen**. You will see this in the images below.
* The **maximum** of about **+1600 HU** corresponds to dense bone.

---
## SECTION 8: Building the 3D Volume — Axial, Coronal and Sagittal Views

We now stack the HU slices (already in geometric order) into a 3D numpy array. With `np.stack`, the array index order is:

```
volume[k, j, i]   ->   (slice, row, column)
```

Using the orientation from Section 5 (i → Right, j → Anterior, k → Superior):

| Plane | How to cut | Fixed index |
|---|---|---|
| **Axial** (transverse) | `volume[k, :, :]` | slice k (fixed height in the body) |
| **Coronal** (front view) | `volume[:, j, :]` | row j (fixed anterior–posterior depth) |
| **Sagittal** (side view) | `volume[:, :, i]` | column i (fixed left–right position) |

**Two practical points:**

1. **Aspect ratio.** For coronal and sagittal views each displayed pixel is *(slice spacing) × (pixel spacing)*, not square. We pass `aspect = slice_spacing / pixel_spacing` to `imshow` so the anatomy has the correct proportions.
2. **Windowing.** HU values range from about −1000 to +2000, but a screen shows only ~256 grey levels, so we display only a **window** of HU values: everything below *centre − width/2* is black and everything above *centre + width/2* is white. Our slices show the upper abdomen, so we use the **soft-tissue window** stored in the header (`WindowCenter`/`WindowWidth` = 40/400). For lungs you would use a **lung window** (centre −600, width 1500).

> **Note:** this example series has only **10 slices** (10 × 0.625 = 6.25 mm of the body). At the physically correct aspect ratio the coronal and sagittal views are therefore very thin strips. We show them both at the correct aspect ratio and **stretched** so you can see their content. With a full scan (hundreds of slices) they would show the whole torso.

In [ ]:
# Cell 8.1: Stacking the Slices into a 3D Volume
# Stack the HU slices into a 3D volume: shape (k, j, i) = (slices, rows, columns)
volume = np.stack(hu_slices, axis=0)
print("Volume shape (slices, rows, columns):", volume.shape)
print("Volume data type                    :", volume.dtype)

# Pick random positions for each plane
k_rand = random.randrange(volume.shape[0])                    # any slice
# For coronal/sagittal stay in the central 50 % so we cut through the body
j_rand = random.randrange(volume.shape[1] // 4, 3 * volume.shape[1] // 4)
i_rand = random.randrange(volume.shape[2] // 4, 3 * volume.shape[2] // 4)
print(f"Random indices -> axial k = {k_rand}, coronal j = {j_rand}, sagittal i = {i_rand}")

# Display window taken from the header (soft-tissue window: centre 40 HU, width 400 HU)
WL, WW = float(first.WindowCenter), float(first.WindowWidth)
vmin, vmax = WL - WW / 2, WL + WW / 2     # HU values mapped to black and white
print(f"Display window: centre {WL:.0f} HU, width {WW:.0f} HU -> [{vmin:.0f}, {vmax:.0f}] HU")

In [ ]:
# Cell 8.2: Axial View with Cut Lines
# Physical sizes needed for correct aspect ratios
dz = gaps.mean()        # spacing between slices (mm)
dy = row_spacing        # spacing between rows (mm)
dx = col_spacing        # spacing between columns (mm)

# --- Axial: one slice, rows vs columns ---------------------------------------
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(volume[k_rand], cmap="gray", vmin=vmin, vmax=vmax, aspect=dy / dx)
ax.set_title(f"Axial slice k = {k_rand} (raw voxel order)")
ax.set_xlabel("column i  →  patient Right")
ax.set_ylabel("row j  →  patient Anterior")
# Show where the coronal (fixed j) and sagittal (fixed i) cuts pass through this slice
ax.axhline(j_rand, color="cyan", lw=1.5, label=f"coronal cut (j = {j_rand})")
ax.axvline(i_rand, color="orange", lw=1.5, label=f"sagittal cut (i = {i_rand})")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8.3: Coronal and Sagittal Reformats
# --- Coronal (fix row j) and sagittal (fix column i) ----------------------------
coronal  = volume[:, j_rand, :]     # shape (slices, columns)
sagittal = volume[:, :, i_rand]     # shape (slices, rows)

fig, axes = plt.subplots(4, 1, figsize=(14, 9),
                         gridspec_kw={"height_ratios": [1, 4, 1, 4]})

# Top: physically correct aspect ratio (one displayed pixel = dz mm tall, dx mm wide).
# origin="lower" puts k = 0 (most inferior slice) at the BOTTOM, so Superior is up.
axes[0].imshow(coronal, cmap="gray", vmin=vmin, vmax=vmax, aspect=dz / dx, origin="lower")
axes[0].set_title(f"Coronal j = {j_rand}: CORRECT aspect ratio (only 6.25 mm tall!)")

# Below: aspect="auto" stretches the image to fill the axes -> anatomy is DISTORTED,
# but we can see the content of the 10 slices.
axes[1].imshow(coronal, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto", origin="lower")
axes[1].set_title(f"Coronal j = {j_rand}: STRETCHED vertically (not to scale)")
axes[1].set_xlabel("column i  →  patient Right")
axes[1].set_ylabel("slice k  →  Superior")

axes[2].imshow(sagittal, cmap="gray", vmin=vmin, vmax=vmax, aspect=dz / dy, origin="lower")
axes[2].set_title(f"Sagittal i = {i_rand}: CORRECT aspect ratio")

axes[3].imshow(sagittal, cmap="gray", vmin=vmin, vmax=vmax, aspect="auto", origin="lower")
axes[3].set_title(f"Sagittal i = {i_rand}: STRETCHED vertically (not to scale)")
axes[3].set_xlabel("row j  →  patient Anterior")
axes[3].set_ylabel("slice k  →  Superior")

for ax in (axes[0], axes[2]):
    ax.set_yticks([])           # the thin strips are too small for tick labels
plt.tight_layout()
plt.show()

**What to notice.** In the stretched views, anatomy changes very little from row to row, because the 10 slices span only 6.25 mm. The vertical structures you see are organs and vessels continuing through the slab. In a full scan these planes would show the whole torso from top to bottom. The grainy texture is image **noise**: the sharp `BONE` reconstruction kernel produces noisy images, and the narrow soft-tissue window makes the noise more visible.

### Displaying in the standard radiological convention

Radiologists view **axial** images *as if looking up from the patient's feet*, with the **patient's Right on the screen's left** and **Anterior at the top**.

In the raw array of this prone scan, column index $i$ increases toward the patient's **Right** (so the patient's Left is on the screen's left) and row index $j$ increases toward **Anterior** (so Posterior, the patient's back, is at the top). To get the standard view we **flip both in-plane axes**.

**Check the anatomy in the radiological view:** the **liver** is on the patient's right (screen left), the **spine** is posterior (bottom), and the curved **scanner table** appears on the *anterior* side because the patient lies face down on it.

This shows why we computed the orientation: without it, you could easily mix up left and right, a serious error in clinical practice.

In [ ]:
# Cell 8.4: Radiological Display Convention
# Flip rows (j) and columns (i) so that: screen-left = patient Right, top = Anterior
volume_radiological = volume[:, ::-1, ::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(volume[k_rand], cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title("Raw array (prone scan)\nleft = patient L, top = Posterior")

axes[1].imshow(volume_radiological[k_rand], cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title("Radiological convention\nleft = patient R, top = Anterior")

# Add orientation labels on the radiological image
h, w = volume_radiological.shape[1:]
for x, y, txt in [(10, h / 2, "R"), (w - 30, h / 2, "L"), (w / 2, 25, "A"), (w / 2, h - 10, "P")]:
    axes[1].text(x, y, txt, color="yellow", fontsize=16, fontweight="bold")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8.5: Same Slice, Three Display Windows
# Bonus: the same axial slice in three display windows.
# Same data, very different appearance. Windowing only changes the DISPLAY.
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
windows = [("Soft-tissue window (C=40, W=400)", 40, 400),
           ("Lung window (C=-600, W=1500)", -600, 1500),
           ("Bone window (C=400, W=1800)", 400, 1800)]
for ax, (title, c, w) in zip(axes, windows):
    ax.imshow(volume_radiological[k_rand], cmap="gray", vmin=c - w / 2, vmax=c + w / 2)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

---
## SECTION 9: From DICOM to NIfTI — Cross-Validation with SimpleITK

### Why another library?
In Sections 1–8 we rebuilt the volume **by hand** to understand every step. In practice, one usually lets a library such as **SimpleITK** (or ITK, nibabel, dcm2niix) read a DICOM series in one call. These libraries perform exactly the steps we implemented:

1. group the files of one series and **sort them geometrically** (by projecting IPP onto the slice normal),
2. apply **RescaleSlope / RescaleIntercept** to obtain HU,
3. store the geometry as **spacing**, **origin** and a 3×3 **direction** matrix.

SimpleITK's geometry is just our affine split into pieces:
$$A_{\text{LPS}} = \begin{bmatrix} D \cdot \operatorname{diag}(\Delta_i, \Delta_j, \Delta_k) & \mathbf{o} \\ \mathbf{0}^\top & 1 \end{bmatrix}$$
where $D$ = `GetDirection()` (columns = direction cosines of the voxel axes), $\Delta$ = `GetSpacing()` and $\mathbf{o}$ = `GetOrigin()`. Like DICOM, **ITK always works in LPS world coordinates**.

### Why NIfTI?
Research tools and deep-learning pipelines rarely work with folders of DICOM files. Instead, the series is converted into a **single file per volume**, most commonly **NIfTI** (`.nii` / `.nii.gz`). A NIfTI file stores the voxel array and one affine, but in **RAS** world coordinates. SimpleITK applies the LPS → RAS flip automatically when writing. The CT and the segmentation masks of Part 2 are stored in exactly this format.

> ⚠️ **Converting DICOM to NIfTI discards most of the header** (patient data, acquisition parameters, windowing, ...). Only the voxel values and the geometry survive. Keep the original DICOM files if you need the metadata later.

In [ ]:
# Cell 9.1: Reading the DICOM Series with SimpleITK
reader = sitk.ImageSeriesReader()
# GDCM finds the files of the series and sorts them geometrically (like Section 4)
sitk_files = reader.GetGDCMSeriesFileNames(DICOM_DIR)
reader.SetFileNames(sitk_files)
ct_dicom_img = reader.Execute()

print("SimpleITK slice order identical to our geometric order:",
      [os.path.basename(f) for f in sitk_files] == sorted_files)
print("Size      (i, j, k)  :", ct_dicom_img.GetSize())
print("Spacing   (i, j, k)  :", np.round(ct_dicom_img.GetSpacing(), 4))
print("Origin    (LPS mm)   :", np.round(ct_dicom_img.GetOrigin(), 4))
print("Direction (3x3, LPS) :\n", np.array(ct_dicom_img.GetDirection()).reshape(3, 3))

In [ ]:
# Cell 9.2: Cross-Validating Our Manual Geometry and HU Conversion
direction = np.array(ct_dicom_img.GetDirection()).reshape(3, 3)
spacing_sitk = np.array(ct_dicom_img.GetSpacing())

# Rebuild the affine from SimpleITK's pieces: A = [D * diag(spacing) | origin]
affine_sitk = np.eye(4)
affine_sitk[:3, :3] = direction @ np.diag(spacing_sitk)
affine_sitk[:3, 3] = ct_dicom_img.GetOrigin()

# SimpleITK has already applied RescaleSlope/RescaleIntercept -> HU, array order [k, j, i]
volume_sitk = sitk.GetArrayFromImage(ct_dicom_img)

checks = {
    "Affine (Section 5) == SimpleITK affine": np.allclose(affine_lps, affine_sitk, atol=1e-3),
    "Voxel size (Section 6) == SimpleITK spacing": np.allclose(voxel_size, spacing_sitk, atol=1e-3),
    "HU volume (Section 8) == SimpleITK voxel values": np.allclose(volume, volume_sitk),
}
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")
assert all(checks.values()), "Manual reconstruction disagrees with SimpleITK!"

# ITK also maps voxel indices to physical points: compare with our affine at the volume centre
centre_index = [s // 2 for s in ct_dicom_img.GetSize()]                # (i, j, k)
p_sitk = ct_dicom_img.TransformIndexToPhysicalPoint(centre_index)
p_ours = (affine_lps @ np.array([*centre_index, 1]))[:3]
print(f"\nCentre voxel {centre_index}: SimpleITK -> {np.round(p_sitk, 3)} mm, our affine -> {np.round(p_ours, 3)} mm")

In [ ]:
# Cell 9.3: Exporting to NIfTI and Re-Reading with nibabel
# Write to a temporary folder (we do not want to modify the course data)
nifti_path = os.path.join(tempfile.gettempdir(), "isa_lab10_dicom_series.nii.gz")
sitk.WriteImage(ct_dicom_img, nifti_path)
print(f"Written: {nifti_path}  ({os.path.getsize(nifti_path) / 1e6:.2f} MB, "
      f"vs. {sum(os.path.getsize(os.path.join(DICOM_DIR, f)) for f in filenames) / 1e6:.2f} MB of DICOM files)")

nii = nib.load(nifti_path)
print("\nAffine stored in the NIfTI file (RAS world):\n", nii.affine)
print("\nEqual to our affine_ras from Cell 5.4 :", np.allclose(nii.affine, affine_ras, atol=1e-3))
print("nibabel orientation codes             :", "".join(nib.aff2axcodes(nii.affine)))
print("Our orientation codes (Section 5)     :", axis_codes(affine_ras, world="RAS"))

# nibabel returns the array in (i, j, k) order, i.e. transposed with respect to SimpleITK's [k, j, i]
print("\nnibabel array shape (i, j, k):", nii.shape, "| SimpleITK array shape (k, j, i):", volume_sitk.shape)

### Interpretation
* SimpleITK sorted the files **in the same geometric order** as our own code, and its spacing, origin, direction and HU values match our manual reconstruction exactly. Our Sections 4–8 are therefore a faithful "look inside" what the library does.
* The NIfTI file stores the affine in **RAS**: it is our `affine_lps` multiplied by `diag(-1, -1, 1, 1)`, exactly as derived in Cell 5.4. The orientation codes do not change, because the voxel axes still point in the same physical directions.
* **Axis order trap:** SimpleITK arrays are indexed `[k, j, i]` (slice first) while nibabel arrays are indexed `[i, j, k]`. Both describe the same volume. Always check `.shape` when switching libraries. We meet this again in Part 2.

---
## SECTION 10.6: Summary of Part 1

| Step | Key takeaway |
|---|---|
| Filenames | Never rely on them for slice order. They may be random UIDs |
| Tags | Fetch by keyword (`ds.PatientID`), by number (`ds[0x0010,0x0020]`), or safely with `ds.get()` |
| Slice order | Sort by projecting `ImagePositionPatient` onto the normal $\mathbf{r}\times\mathbf{c}$ computed from `ImageOrientationPatient` |
| Affine | Built from IOP (directions), PixelSpacing + slice spacing (scales) and IPP of the first slice (translation) |
| Orientation | The DICOM world is always **LPS**. The voxel array here is ordered **RAS** because the patient was scanned **prone** |
| Spacing | 0.703 × 0.703 mm in-plane. Slice spacing 0.625 mm ≠ slice thickness 1.25 mm (50 % overlap) |
| Intensities | Apply `RescaleSlope`/`RescaleIntercept` → HU. Values around −2048 are outside the field of view, not tissue |
| Visualisation | Use the correct aspect ratio and an appropriate HU window, and check left/right |
| Content check | Look at the images and the z-coordinates. A "chest" series may contain abdominal slices, as here |
| SimpleITK / NIfTI | Libraries do the same sorting, rescaling and affine construction. NIfTI stores the affine in **RAS** and drops most of the DICOM header |

---
# PART 2: Evaluating Image Segmentation — Global vs. Lesion-wise Metrics

---

### Learning objectives of Part 2
By the end of this part you should be able to:

1. Load 3D medical images (NIfTI, `.nii.gz`, see Section 9) with **SimpleITK** and convert them to **NumPy** arrays.
2. Visualise a CT scan slice-by-slice with a segmentation mask overlaid, using an interactive slider.
3. Compute the four most common segmentation metrics: **Dice**, **Precision**, **Recall** and **HD95**.
4. Understand **why global metrics can be misleading** when an image contains several lesions, or when a mask is not what you expect it to be (e.g. not binary).
5. Implement a **lesion-matching** procedure and compute **lesion-wise (instance-wise)** metrics.

**Data.** In the folder `example_data/` (downloaded in Section 0.2) we have:

| File | Content |
|---|---|
| `ct.nii.gz` | The CT image (intensities in Hounsfield Units, HU) |
| `reference.nii.gz` | The **reference** (ground-truth) lesion segmentation, drawn by an expert |
| `pred1.nii.gz` | Prediction of algorithm 1 |
| `pred2.nii.gz` | Prediction of algorithm 2 |
| `pred3.nii.gz` | Prediction of algorithm 3 |

Our task is to answer a simple question: **which algorithm segments the lesions best?** As we will see, the answer depends a lot on *how* we measure.

---
## SECTION 11: Loading the Images

A NIfTI file contains two things:

1. The **voxel values** (a 3D array of numbers).
2. The **geometry**: voxel *spacing* (size of a voxel in mm), *origin* (position of the first voxel in the scanner coordinate system) and *direction* (orientation of the axes).

SimpleITK keeps both together in an `sitk.Image`. To do arithmetic we convert it into a NumPy array with `sitk.GetArrayFromImage`.

> ⚠️ **Axis order!** SimpleITK reports sizes as `(x, y, z)`, but the NumPy array is indexed as `[z, y, x]`. So `array[k]` is the **k-th axial slice**.

We keep the spacing because distances (HD95) must be measured in **millimetres**, not in voxels: voxels are often not cubic (here 0.65 × 0.65 × 1.5 mm).

> 🔗 This is exactly the format we produced from DICOM in Section 9. The `[z, y, x]` NumPy order corresponds to the `[k, j, i]` order of Part 1.

In [ ]:
# Cell 11.1: Loading the CT, Reference and Predictions
# Names of the three algorithms we want to evaluate
PRED_NAMES = ["pred1", "pred2", "pred3"]


def load_image(name):
    # Read a NIfTI file and return (SimpleITK image, NumPy array).
    path = os.path.join(DATA_DIR, f"{name}.nii.gz")   # e.g. ./example_data/ct.nii.gz
    image = sitk.ReadImage(path)                      # SimpleITK image (values + geometry)
    array = sitk.GetArrayFromImage(image)             # NumPy array with axes [z, y, x]
    return image, array


# --- CT image ------------------------------------------------------------------
ct_img, ct = load_image("ct")

# --- Reference (ground truth) ---------------------------------------------------
ref_img, ref = load_image("reference")

# --- The three predictions, stored in a dictionary {name: array} ---------------
preds = {}
for name in PRED_NAMES:
    _, preds[name] = load_image(name)

# Voxel spacing in (x, y, z) order (SimpleITK convention) ...
spacing_xyz = ct_img.GetSpacing()
# ... and reversed to (z, y, x) so that it matches the NumPy axis order.
spacing_zyx = spacing_xyz[::-1]

print("CT size (x, y, z)     :", ct_img.GetSize())
print("NumPy shape (z, y, x) :", ct.shape)
print("Spacing (x, y, z) mm  :", spacing_xyz)
print("CT intensity range HU :", ct.min(), "to", ct.max())

### 11.2 Sanity checks — always look at your data before computing anything!

Before evaluating, we check two things:

* **Geometry**: all images must have the same size, spacing, origin and direction; otherwise voxel `[z, y, x]` in one image would not correspond to the same physical point in another.
* **Labels**: which values occur in each mask? A binary mask should only contain `0` (background) and `1` (lesion).

In [ ]:
# Cell 11.2: Geometry and Label Sanity Checks
def same_geometry(img_a, img_b, tol=1e-4):
    # True if two SimpleITK images live on exactly the same voxel grid.
    return (img_a.GetSize() == img_b.GetSize()
            and np.allclose(img_a.GetSpacing(), img_b.GetSpacing(), atol=tol)
            and np.allclose(img_a.GetOrigin(), img_b.GetOrigin(), atol=tol)
            and np.allclose(img_a.GetDirection(), img_b.GetDirection(), atol=tol))


# Check every mask against the CT grid
for name in ["reference"] + PRED_NAMES:
    img, _ = load_image(name)
    print(f"{name:10s} same grid as CT: {same_geometry(ct_img, img)}")

print()

# List the unique values (labels) present in each mask, with their voxel counts
for name, arr in [("reference", ref)] + list(preds.items()):
    values, counts = np.unique(arr, return_counts=True)
    print(f"{name:10s} dtype={str(arr.dtype):8s} labels -> "
          + ", ".join(f"{v:g}: {c} voxels" for v, c in zip(values, counts)))

Notice anything unusual? Keep the output above in mind; we will return to it in **Section 15**.

For now, we turn the reference into a clean boolean array (`True` = lesion). The reference is stored as `float32` with values 0.0/1.0, so `ref > 0` is a safe way to binarise it.

In [ ]:
# Cell 11.3: Binarising the Reference
# Boolean reference mask: True where the expert drew a lesion
ref_bin = ref > 0
print("Reference lesion volume:", ref_bin.sum(), "voxels =",
      round(ref_bin.sum() * np.prod(spacing_xyz) / 1000, 2), "mL")

---
## SECTION 12: Interactive Visualisation — CT + Reference Mask (Axial View)

### 12.1 CT windowing
CT values range from about −1000 HU (air) to +1000 HU (bone) and beyond. A screen can only show 256 grey levels, and our eyes distinguish far fewer, so we **window** the image: only intensities in `[level − width/2, level + width/2]` are spread over black → white; everything below is black and everything above is white.
A typical **soft-tissue window** is *level = 40 HU, width = 400 HU* (the same window as in Section 8).

### 12.2 Overlay
To overlay a mask, we hide the background voxels with a *masked array* (`np.ma.masked_where`) so that only lesion voxels are coloured, and draw them semi-transparently (`alpha`) on top of the CT.

### 12.3 Slider
`ipywidgets.interact` connects a slider to a plotting function: every time you move the slider, the function is called again with the new slice index.

> 💡 The slider starts at the slice that contains the most reference-lesion voxels, so you see something interesting immediately.

> 🖥️ The sliders need a live kernel. On a static view of the notebook (e.g. on GitHub) they are not shown; a static comparison figure is provided in Cell 13.4.

In [ ]:
# Cell 12.1: Interactive Overlay Viewer
# Colours for mask labels: index 0 = background (never drawn), 1 = red, 2 = cyan, ...
LABEL_CMAP = ListedColormap(["black", "red", "cyan", "orange", "magenta"])


def window_ct(ct_slice, level=40, width=400):
    # Clip a CT slice to the window [level - width/2, level + width/2].
    lo, hi = level - width / 2, level + width / 2
    return np.clip(ct_slice, lo, hi)


def show_overlay(ct_vol, mask_vol, title, contour_vol=None, level=40, width=400):
    # Interactive axial viewer: CT with a (possibly multi-label) mask overlaid.
    #   ct_vol      : 3D CT array [z, y, x]
    #   mask_vol    : 3D integer/bool mask with the same shape (0 = background)
    #   title       : text shown above the image
    #   contour_vol : optional second binary mask drawn as a yellow contour
    #                 (useful to compare a prediction with the reference)

    mask_vol = np.asarray(mask_vol).astype(np.uint8)   # make sure labels are integers
    n_slices = ct_vol.shape[0]                          # number of axial slices

    # Start at the slice with the largest lesion area (of the contour mask if given)
    focus = contour_vol if contour_vol is not None else mask_vol
    start = int(np.argmax((focus > 0).sum(axis=(1, 2))))

    def draw(z, alpha):
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        # ---- Left panel: CT only --------------------------------------------
        axes[0].imshow(window_ct(ct_vol[z], level, width), cmap="gray")
        axes[0].set_title(f"CT, axial slice {z}/{n_slices - 1}")

        # ---- Right panel: CT + mask overlay ----------------------------------
        axes[1].imshow(window_ct(ct_vol[z], level, width), cmap="gray")
        # Hide background (label 0) so only lesion voxels are coloured
        overlay = np.ma.masked_where(mask_vol[z] == 0, mask_vol[z])
        axes[1].imshow(overlay, cmap=LABEL_CMAP, vmin=0, vmax=LABEL_CMAP.N - 1,
                       alpha=alpha, interpolation="nearest")
        # Optional reference contour in yellow
        if contour_vol is not None and contour_vol[z].any():
            axes[1].contour(contour_vol[z] > 0, levels=[0.5], colors="yellow", linewidths=1)
        # Which labels are visible on this slice? (helps to spot non-binary masks)
        labels_here = [int(v) for v in np.unique(mask_vol[z]) if v != 0]
        axes[1].set_title(f"{title} | labels on slice: {labels_here}")

        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    # Two sliders: slice index and overlay transparency
    interact(draw,
             z=widgets.IntSlider(value=start, min=0, max=n_slices - 1, step=1,
                                 description="Axial slice", continuous_update=False,
                                 layout=widgets.Layout(width="600px")),
             alpha=widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                       description="Opacity", continuous_update=False))

In [ ]:
# Cell 12.2: Reference Mask on the CT
# Reference mask (red) on top of the CT
show_overlay(ct, ref_bin, title="Reference")

**Things to try:**
* Scroll through the slices. How many separate lesions do you see? Are they all the same size?
* Set the opacity to 0 and 1 to compare the lesion boundary with the underlying anatomy.

---
## SECTION 13: Visualising the Three Predictions

We now use the very same viewer for each algorithm. The prediction is shown in colour, and the **reference is drawn as a yellow contour**, so you can directly see:

* **Over-segmentation** – coloured voxels outside the yellow contour (false positives).
* **Under-segmentation** – parts inside the yellow contour that are not coloured (false negatives).
* **Spurious / missed lesions** – coloured blobs where there is no yellow contour at all, or yellow contours with no colour inside.

> Each mask is passed to the viewer **as it is stored on disk** (we have not modified the predictions). Label 1 is shown in **red**, label 2 (if present) in **cyan**.

### 13.1 Prediction 1

In [ ]:
# Cell 13.1: Prediction 1
show_overlay(ct, preds["pred1"], title="Prediction 1", contour_vol=ref_bin)

### 13.2 Prediction 2

In [ ]:
# Cell 13.2: Prediction 2
show_overlay(ct, preds["pred2"], title="Prediction 2", contour_vol=ref_bin)

### 13.3 Prediction 3

In [ ]:
# Cell 13.3: Prediction 3
show_overlay(ct, preds["pred3"], title="Prediction 3", contour_vol=ref_bin)

### 13.4 Static side-by-side comparison
The same information in a single non-interactive figure, on the slice with the largest reference area. Each prediction is drawn exactly as stored on disk (label 1 red, label 2 cyan), with the reference contour in yellow.

In [ ]:
# Cell 13.4: Static Comparison of All Predictions
z_show = int(np.argmax(ref_bin.sum(axis=(1, 2))))      # slice with the largest reference area

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
masks = [("Reference", ref_bin.astype(np.uint8))] + [(name, preds[name].astype(np.uint8)) for name in PRED_NAMES]
for ax, (title, mask) in zip(axes, masks):
    ax.imshow(window_ct(ct[z_show]), cmap="gray")
    ax.imshow(np.ma.masked_where(mask[z_show] == 0, mask[z_show]), cmap=LABEL_CMAP,
              vmin=0, vmax=LABEL_CMAP.N - 1, alpha=0.5, interpolation="nearest")
    if title != "Reference":
        ax.contour(ref_bin[z_show], levels=[0.5], colors="yellow", linewidths=1)
    ax.set_title(f"{title} (slice {z_show})", fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()

**Question for you:** Do you see any colour in prediction 3 that does *not* appear in the other predictions? What could it mean?

---
## SECTION 14: Global Segmentation Metrics

Let **R** be the set of reference (ground-truth) lesion voxels and **P** the set of predicted lesion voxels. Each voxel falls into one of four categories:

| | Reference = lesion | Reference = background |
|---|---|---|
| **Prediction = lesion** | True Positive (TP) | False Positive (FP) |
| **Prediction = background** | False Negative (FN) | True Negative (TN) |

From these counts we define:

* **Dice similarity coefficient** — overlap between R and P (0 = no overlap, 1 = perfect):
  $$\text{Dice} = \frac{2\,|R \cap P|}{|R| + |P|} = \frac{2\,TP}{2\,TP + FP + FN}$$
* **Precision** (positive predictive value) — *of everything predicted as lesion, how much is really lesion?* Penalises **over-segmentation**:
  $$\text{Precision} = \frac{TP}{TP + FP}$$
* **Recall** (sensitivity) — *of all real lesion voxels, how many did we find?* Penalises **under-segmentation**:
  $$\text{Recall} = \frac{TP}{TP + FN}$$
* **HD95 — 95th-percentile Hausdorff distance** (in mm). Dice/Precision/Recall only *count* voxels; they do not care *where* the errors are. HD95 is a **boundary distance**:
  1. extract the surface (boundary voxels) of R and of P;
  2. for every surface point of P, measure the distance to the closest surface point of R, and vice versa;
  3. HD95 is the 95th percentile of these distances (we take the larger of the two directions). Using the 95th percentile instead of the maximum makes it robust to a handful of outlier voxels.

  Lower is better; 0 mm means the boundaries coincide.

> Note that TN does not appear in any formula: in a CT volume with ~33 million voxels, almost all are background, so accuracy-like metrics that use TN would always be ≈ 1 and therefore useless.

**"Global"** means we treat *all* lesion voxels of the image as one big object, ignoring which lesion they belong to.

### 14.1 Implementing the metrics

**Edge cases.** What if a mask is empty?
* Both empty → the prediction is perfect: Dice = Precision = Recall = 1, HD95 = 0.
* Only one empty → the overlap metrics are 0 (or undefined), and HD95 is **undefined** (no surface to measure to). We return `NaN` (Not a Number) for undefined values.

**HD95 implementation.** We use a *Euclidean distance transform* (EDT): `ndimage.distance_transform_edt(x, sampling=spacing)` returns, for every non-zero voxel of `x`, the distance **in mm** to the nearest zero voxel. If we feed it "everything except the surface of R", we get the distance from every voxel to the surface of R. To save time and memory we first crop both masks to a bounding box around them: this does not change the result, because every surface voxel lies inside the box.

In [ ]:
# Cell 14.1: First-Principles Metric Implementations
def overlap_metrics(gt, pred):
    # Dice, precision and recall between two boolean masks of the same shape.
    gt = gt.astype(bool)
    pred = pred.astype(bool)

    tp = np.logical_and(gt, pred).sum()     # lesion in both
    fp = np.logical_and(~gt, pred).sum()    # predicted lesion, but background in reference
    fn = np.logical_and(gt, ~pred).sum()    # reference lesion that was missed

    # Both masks empty: nothing to find and nothing found -> perfect agreement
    if gt.sum() == 0 and pred.sum() == 0:
        return 1.0, 1.0, 1.0

    dice = 2 * tp / (2 * tp + fp + fn)
    # Precision is undefined if nothing was predicted; recall if there is no lesion
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return float(dice), float(precision), float(recall)


def surface(mask):
    # Boundary voxels of a boolean mask: voxels that are in the mask
    # but disappear after one step of erosion.
    eroded = ndimage.binary_erosion(mask, structure=ndimage.generate_binary_structure(3, 1))
    return np.logical_and(mask, ~eroded)


def hd95(gt, pred, spacing):
    # 95th-percentile (symmetric) Hausdorff distance in mm.
    #   spacing must be in NumPy (z, y, x) order.
    gt = gt.astype(bool)
    pred = pred.astype(bool)

    if not gt.any() and not pred.any():
        return 0.0          # both empty -> identical
    if not gt.any() or not pred.any():
        return np.nan       # one is empty -> distance undefined

    # ---- 1. Crop to the bounding box of both masks (+1 voxel margin) ---------
    coords = np.argwhere(gt | pred)                       # (N, 3) indices of foreground
    lo = np.maximum(coords.min(axis=0) - 1, 0)
    hi = np.minimum(coords.max(axis=0) + 2, gt.shape)
    box = tuple(slice(l, h) for l, h in zip(lo, hi))
    gt, pred = gt[box], pred[box]

    # ---- 2. Surface voxels ----------------------------------------------------
    s_gt, s_pred = surface(gt), surface(pred)

    # ---- 3. Distance (mm) from every voxel to the nearest surface voxel ------
    # distance_transform_edt measures the distance to the nearest ZERO,
    # so we invert the surface masks (surface = 0, everything else = 1).
    dist_to_gt = ndimage.distance_transform_edt(~s_gt, sampling=spacing)
    dist_to_pred = ndimage.distance_transform_edt(~s_pred, sampling=spacing)

    # ---- 4. Directed distances, read off at the surface voxels ---------------
    d_pred_to_gt = dist_to_gt[s_pred]       # each predicted surface point -> reference
    d_gt_to_pred = dist_to_pred[s_gt]       # each reference surface point -> prediction

    # ---- 5. Symmetric HD95: the worse of the two directed 95th percentiles ---
    return float(max(np.percentile(d_pred_to_gt, 95), np.percentile(d_gt_to_pred, 95)))


def all_metrics(gt, pred, spacing):
    # Convenience wrapper returning a dict with the four metrics.
    dice, precision, recall = overlap_metrics(gt, pred)
    return {"Dice": dice, "Precision": precision, "Recall": recall,
            "HD95 (mm)": hd95(gt, pred, spacing)}

**Always test your metric code on a toy example where you know the answer!**

Below, two 10×10×10 cubes are shifted by 2 voxels along *x*. The overlap is 8×10×10 = 800 voxels, so Dice = 2·800/(1000+1000) = 0.8. With 1 mm spacing, the two boundaries are 2 mm apart where they differ, so HD95 should be 2 mm.

In [ ]:
# Cell 14.2: Unit Test of the Metrics on a Toy Example
toy_gt = np.zeros((20, 20, 20), dtype=bool)
toy_pred = np.zeros((20, 20, 20), dtype=bool)
toy_gt[5:15, 5:15, 5:15] = True      # 10x10x10 cube
toy_pred[5:15, 5:15, 7:17] = True    # same cube shifted by 2 voxels in x

toy_result = all_metrics(toy_gt, toy_pred, spacing=(1.0, 1.0, 1.0))
print(toy_result)
# Expected: Dice = 0.8, Precision = 0.8, Recall = 0.8, HD95 = 2.0

# Automated checks, including the edge cases described above
assert np.isclose(toy_result["Dice"], 0.8) and np.isclose(toy_result["Precision"], 0.8) and np.isclose(toy_result["Recall"], 0.8)
assert np.isclose(toy_result["HD95 (mm)"], 2.0)
assert all_metrics(toy_gt, toy_gt, (1.0, 1.0, 1.0)) == {"Dice": 1.0, "Precision": 1.0, "Recall": 1.0, "HD95 (mm)": 0.0}
empty = np.zeros_like(toy_gt)
assert overlap_metrics(empty, empty) == (1.0, 1.0, 1.0) and hd95(empty, empty, (1.0, 1.0, 1.0)) == 0.0
assert np.isnan(hd95(toy_gt, empty, (1.0, 1.0, 1.0)))
print("All metric unit tests passed.")

### 14.2 Computing the global metrics for each algorithm

A frequent (and dangerous) shortcut is to assume that **foreground = label 1**, i.e. to binarise a prediction with `pred == 1`. This is exactly what many evaluation scripts do. Let's use this convention, as a "naive" evaluator would, and see what happens.

In [ ]:
# Cell 14.3: Naive Global Metrics (pred == 1)
global_rows = []
for name in PRED_NAMES:
    # Naive binarisation: only voxels with value 1 count as lesion
    pred_bin_naive = preds[name] == 1
    row = all_metrics(ref_bin, pred_bin_naive, spacing_zyx)
    row["Algorithm"] = name
    global_rows.append(row)

# Put the results in a table, one row per algorithm
global_naive = pd.DataFrame(global_rows).set_index("Algorithm")
global_naive

### 14.3 Reading the table

At first glance, the table gives us a clear ranking. But is it trustworthy? Before drawing conclusions we should ask two questions:

1. **What does "global" hide?** The image contains *several separate lesions* of very different sizes.
2. **Did we evaluate what we think we evaluated?** Remember the label check in Section 11.2.

The next section discusses both.

---
## SECTION 15: The Problems with Naive Global Metrics

### Problem 1 — Several lesions (instances) are merged into one object

Global metrics pool all lesion voxels together. Consequences:

* **Large lesions dominate.** A big lesion with 36 000 voxels contributes far more to Dice than a small lesion of 200 voxels. An algorithm can completely **miss every small lesion** and still obtain a high global Dice, because the large one is segmented well. Clinically, however, missing a small lesion (e.g. an early metastasis) can be the most important error!
* **No detection information.** Global metrics cannot tell us *how many* lesions were found, missed (false-negative lesions) or hallucinated (false-positive lesions).
* **HD95 becomes a "distance between far-away lesions".** One false-positive blob far away from any real lesion inflates the global HD95, but the number says nothing about how well each individual lesion boundary was drawn.

**Solution:** split the masks into individual lesions (*connected components*), **match** predicted lesions to reference lesions, and compute the metrics **per lesion**. This is called **lesion-wise** or **instance-wise** evaluation.

### Problem 2 — Prediction 3 is *not* a binary mask

The label check in Section 11.2 showed that `pred3` contains **two labels, 1 and 2**, while the reference and the other predictions only contain 0 and 1. (In the viewer, label 2 appears in cyan.) Algorithm 3 probably outputs a second class (e.g. two lesion sub-types or two tissue compartments), but **our reference only defines a single "lesion" class**.

Using the `pred == 1` convention, every label-2 voxel was silently treated as **background**: inside the reference these voxels became false negatives, so the global recall and Dice of algorithm 3 were **underestimated**. Note that no error was raised — the code ran happily and produced a wrong number. Different tools handle this differently (`pred == 1`, `pred > 0`, `pred.astype(bool)`, or treating each label separately), so the same prediction can get very different scores depending on the script.

**Solution:** since the task is binary (lesion vs. background), merge all non-zero labels into one lesion class: `pred3_bin = pred3 > 0`.

Let's verify how many label-2 voxels actually lie inside the reference:

In [ ]:
# Cell 15.1: Where Are the Label-2 Voxels of Prediction 3?
p3 = preds["pred3"]
n_label2 = (p3 == 2).sum()                           # all voxels with label 2
n_label2_in_ref = np.logical_and(p3 == 2, ref_bin).sum()  # label-2 voxels inside the reference

print(f"Label-2 voxels in pred3                : {n_label2}")
print(f"... of which inside the reference lesion: {n_label2_in_ref} "
      f"({100 * n_label2_in_ref / n_label2:.1f} %)")
print(f"Share of the reference covered by label 2: {100 * n_label2_in_ref / ref_bin.sum():.1f} %")

---
## SECTION 16: Fix 1 — Binarise Prediction 3

We merge all non-zero labels into one lesion class. To be consistent, we binarise **all** predictions the same way (`> 0`); for pred1 and pred2 this changes nothing because they only contain 0 and 1.

In [ ]:
# Cell 16.1: Binarising All Predictions
# Binary (boolean) versions of every prediction: any non-zero label = lesion
preds_bin = {name: preds[name] > 0 for name in PRED_NAMES}

# Check: each binary prediction now only contains False/True
for name, arr in preds_bin.items():
    print(f"{name}: unique values = {np.unique(arr)}, lesion voxels = {arr.sum()}")

In [ ]:
# Cell 16.2: Prediction 3 After Binarisation
# Look at prediction 3 again: now it is a single (red) label
show_overlay(ct, preds_bin["pred3"], title="Prediction 3 (binarised)", contour_vol=ref_bin)

Let's recompute the global metrics with the correctly binarised predictions and compare with the naive table:

In [ ]:
# Cell 16.3: Naive vs. Binarised Global Metrics
global_rows = []
for name in PRED_NAMES:
    row = all_metrics(ref_bin, preds_bin[name], spacing_zyx)
    row["Algorithm"] = name
    global_rows.append(row)
global_fixed = pd.DataFrame(global_rows).set_index("Algorithm")

# Show both tables side by side (multi-level columns)
pd.concat({"Naive global (pred == 1)": global_naive,
           "Global (binarised, pred > 0)": global_fixed}, axis=1)

Only the `pred3` row changes, and it changes a lot. The global metrics are now at least computed on the right masks — but Problem 1 (multiple lesions) remains.

---
## SECTION 17: Fix 2 — Lesion Matching

### 17.1 Step A: find the individual lesions (connected components)

A **connected component** is a group of foreground voxels that touch each other. `scipy.ndimage.label` gives every component a unique integer ID (1, 2, 3, ...).

We must choose a **connectivity** — which neighbours count as "touching":
* 6-connectivity: only face neighbours;
* 26-connectivity: face, edge and corner neighbours (`np.ones((3, 3, 3))`).

We use 26-connectivity, the most common choice for 3D lesions. Keep in mind that this choice (and e.g. removing tiny components) can change the number of lesions.

In [ ]:
# Cell 17.1: Connected Components (Lesion Instances)
# 3x3x3 block of ones = 26-connectivity
CONNECTIVITY = np.ones((3, 3, 3), dtype=bool)


def get_instances(mask):
    # Split a binary mask into connected components.
    # Returns (labelled array, number of components); background = 0.
    labelled, n = ndimage.label(mask, structure=CONNECTIVITY)
    return labelled, n


# Instances of the reference
ref_inst, n_ref = get_instances(ref_bin)
# Instances of each (binarised) prediction
pred_inst = {name: get_instances(preds_bin[name]) for name in PRED_NAMES}

print(f"reference: {n_ref} lesions, sizes (voxels) = {np.bincount(ref_inst.ravel())[1:].tolist()}")
for name, (_, n) in pred_inst.items():
    print(f"{name:9s}: {n} predicted lesions")

In [ ]:
# Cell 17.2: Visualising the Reference Instances
# Visualise the reference instances: each lesion gets its own colour
def show_instances(ct_vol, inst_vol, title):
    n_slices = ct_vol.shape[0]
    start = int(np.argmax((inst_vol > 0).sum(axis=(1, 2))))
    cmap = plt.get_cmap("tab20")                     # 20 distinct colours

    def draw(z):
        plt.figure(figsize=(7, 6))
        plt.imshow(window_ct(ct_vol[z]), cmap="gray")
        overlay = np.ma.masked_where(inst_vol[z] == 0, inst_vol[z] % 20)
        plt.imshow(overlay, cmap=cmap, vmin=0, vmax=19, alpha=0.6, interpolation="nearest")
        # Write each lesion ID at the centre of the lesion on this slice
        for lesion_id in np.unique(inst_vol[z]):
            if lesion_id == 0:
                continue
            yy, xx = np.nonzero(inst_vol[z] == lesion_id)
            plt.text(xx.mean(), yy.mean(), str(lesion_id), color="white",
                     fontsize=11, ha="center", va="center", fontweight="bold")
        plt.title(f"{title} - slice {z}")
        plt.axis("off")
        plt.show()

    interact(draw, z=widgets.IntSlider(value=start, min=0, max=n_slices - 1,
                                       description="Axial slice", continuous_update=False,
                                       layout=widgets.Layout(width="600px")))


show_instances(ct, ref_inst, "Reference lesion instances")

### 17.2 Step B: match predicted lesions to reference lesions

Now we must decide **which predicted lesion corresponds to which reference lesion**. A conventional and widely used recipe (used e.g. in panoptic-quality style evaluation) is:

1. Compute the **Intersection over Union (IoU)** between every reference lesion *i* and every predicted lesion *j*:
   $$\text{IoU}_{ij} = \frac{|R_i \cap P_j|}{|R_i \cup P_j|}$$
   This gives an IoU matrix of size *(number of reference lesions) × (number of predicted lesions)*.
2. Solve a **one-to-one assignment** with the **Hungarian algorithm** (`scipy.optimize.linear_sum_assignment`), which finds the pairing that maximises the total IoU, with each lesion used at most once.
3. Keep only pairs whose IoU exceeds a **threshold** (here `IOU_THRESHOLD = 0.1`). Assigned pairs below the threshold are considered unmatched — they barely overlap.

Each lesion then ends up in one of three groups:

| Group | Meaning |
|---|---|
| **TP lesion** (matched pair) | reference lesion that was detected |
| **FN lesion** (unmatched reference) | reference lesion that was **missed** |
| **FP lesion** (unmatched prediction) | predicted lesion with **no counterpart** in the reference |

> 💡 The threshold is a design choice. `IoU > 0.5` is used in panoptic quality (and guarantees a unique match even without the Hungarian algorithm); medical challenges often use a lenient threshold (any overlap, or IoU > 0.1) because small lesions are hard to delineate precisely. Try changing the value and see how the results change!

In [ ]:
# Cell 17.3: IoU Matrix and Hungarian Matching
IOU_THRESHOLD = 0.1   # minimum IoU for a reference/prediction pair to be a match


def iou_matrix(gt_inst, n_gt, pr_inst, n_pr):
    # IoU between every reference instance (rows) and every predicted instance (columns).
    # Count, for every (gt_id, pred_id) combination, the number of shared voxels.
    # Trick: encode each voxel's pair of IDs as a single number gt_id * (n_pr + 1) + pr_id
    # and count them all at once with np.bincount (much faster than a double loop).
    fg = (gt_inst > 0) | (pr_inst > 0)                   # only voxels that are lesion somewhere
    pair_code = gt_inst[fg].astype(np.int64) * (n_pr + 1) + pr_inst[fg]
    counts = np.bincount(pair_code, minlength=(n_gt + 1) * (n_pr + 1))
    counts = counts.reshape(n_gt + 1, n_pr + 1)          # counts[i, j] = |R_i ∩ P_j|  (0 = background)

    intersection = counts[1:, 1:]                        # drop background row/column
    size_gt = np.bincount(gt_inst.ravel(), minlength=n_gt + 1)[1:]   # |R_i|
    size_pr = np.bincount(pr_inst.ravel(), minlength=n_pr + 1)[1:]   # |P_j|
    union = size_gt[:, None] + size_pr[None, :] - intersection        # |R_i ∪ P_j|
    return intersection / union


def match_lesions(gt_inst, n_gt, pr_inst, n_pr, iou_threshold=IOU_THRESHOLD):
    # One-to-one lesion matching with the Hungarian algorithm.
    # Returns: list of matched (gt_id, pred_id, iou), list of missed gt_ids,
    #          list of false-positive pred_ids. IDs start at 1.
    if n_gt == 0 or n_pr == 0:                           # nothing to match
        return [], list(range(1, n_gt + 1)), list(range(1, n_pr + 1))

    iou = iou_matrix(gt_inst, n_gt, pr_inst, n_pr)

    # linear_sum_assignment MINIMISES a cost, so we use cost = -IoU to MAXIMISE IoU
    rows, cols = linear_sum_assignment(-iou)

    matches = []
    for r, c in zip(rows, cols):
        if iou[r, c] > iou_threshold:                    # accept only sufficient overlap
            matches.append((r + 1, c + 1, float(iou[r, c])))   # +1: back to instance IDs

    matched_gt = {m[0] for m in matches}
    matched_pr = {m[1] for m in matches}
    missed = [i for i in range(1, n_gt + 1) if i not in matched_gt]          # FN lesions
    false_pos = [j for j in range(1, n_pr + 1) if j not in matched_pr]       # FP lesions
    return matches, missed, false_pos

In [ ]:
# Cell 17.4: Matching Results per Algorithm
# Run the matching for each algorithm and print a summary
matching = {}
for name in PRED_NAMES:
    p_inst, n_p = pred_inst[name]
    matches, missed, false_pos = match_lesions(ref_inst, n_ref, p_inst, n_p)
    matching[name] = (matches, missed, false_pos)

    print(f"===== {name} =====")
    print(f"  TP lesions (matched): {len(matches)}  |  FN lesions (missed): {len(missed)}"
          f"  |  FP lesions: {len(false_pos)}")
    for g, p, iou in matches:
        print(f"    reference lesion {g:2d} <-> predicted lesion {p:2d}   IoU = {iou:.2f}")
    print(f"  missed reference lesions: {missed}")

### 17.3 Detection metrics (lesion level)

Once lesions are matched we can also measure **detection** performance, which is impossible with global metrics:

$$\text{Lesion precision} = \frac{TP_{les}}{TP_{les} + FP_{les}}, \qquad \text{Lesion recall (sensitivity)} = \frac{TP_{les}}{TP_{les} + FN_{les}}, \qquad \text{Lesion F1} = \frac{2\,TP_{les}}{2\,TP_{les} + FP_{les} + FN_{les}}$$

In [ ]:
# Cell 17.5: Lesion-Level Detection Metrics
det_rows = []
for name in PRED_NAMES:
    matches, missed, false_pos = matching[name]
    tp, fn, fp = len(matches), len(missed), len(false_pos)
    det_rows.append({
        "Algorithm": name,
        "Ref. lesions": n_ref,
        "Pred. lesions": pred_inst[name][1],
        "TP lesions": tp,
        "FN lesions": fn,
        "FP lesions": fp,
        "Lesion precision": tp / (tp + fp) if tp + fp else np.nan,
        "Lesion recall": tp / (tp + fn) if tp + fn else np.nan,
        "Lesion F1": 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else np.nan,
    })
detection = pd.DataFrame(det_rows).set_index("Algorithm")
detection

---
## SECTION 18: Lesion-wise Metrics

For every lesion we now compute Dice, Precision, Recall and HD95 **between the two matched components only** (`ref_inst == g` vs. `pred_inst == p`), not between the full masks.

How do we score lesions without a partner?

| Lesion type | Dice | Precision | Recall | HD95 |
|---|---|---|---|---|
| TP (matched) | computed | computed | computed | computed |
| FN (missed reference lesion) | 0 | 0 | 0 | undefined (`NaN`) |
| FP (spurious predicted lesion) | 0 | 0 | 0 | undefined (`NaN`) |

Setting Dice/Precision/Recall to 0 for unmatched lesions means that missing or hallucinating a lesion is **penalised**, regardless of its size. HD95 is undefined when one mask is empty; we leave it as `NaN` and average it over matched lesions only (another common choice is to assign a large penalty value, e.g. the image diagonal). Because of this, **always report the lesion-wise HD95 together with the detection counts.**

In [ ]:
# Cell 18.1: Per-Lesion Metric Tables
def lesionwise_metrics(gt_inst, pr_inst, matches, missed, false_pos, spacing):
    # One row per lesion (TP, FN and FP lesions) with the four metrics.
    rows = []

    # ---- Matched pairs: real per-lesion metrics -------------------------------
    for g, p, iou in matches:
        gt_lesion = gt_inst == g        # boolean mask of reference lesion g only
        pr_lesion = pr_inst == p        # boolean mask of predicted lesion p only
        m = all_metrics(gt_lesion, pr_lesion, spacing)
        m.update({"Type": "TP", "Ref. ID": g, "Pred. ID": p,
                  "Ref. size (vox)": int(gt_lesion.sum()), "IoU": iou})
        rows.append(m)

    # ---- Missed reference lesions (FN) -------------------------------------
    for g in missed:
        rows.append({"Type": "FN", "Ref. ID": g, "Pred. ID": None,
                     "Ref. size (vox)": int((gt_inst == g).sum()), "IoU": 0.0,
                     "Dice": 0.0, "Precision": 0.0, "Recall": 0.0, "HD95 (mm)": np.nan})

    # ---- Spurious predicted lesions (FP) ------------------------------------
    for p in false_pos:
        rows.append({"Type": "FP", "Ref. ID": None, "Pred. ID": p,
                     "Ref. size (vox)": 0, "IoU": 0.0,
                     "Dice": 0.0, "Precision": 0.0, "Recall": 0.0, "HD95 (mm)": np.nan})

    cols = ["Type", "Ref. ID", "Pred. ID", "Ref. size (vox)", "IoU",
            "Dice", "Precision", "Recall", "HD95 (mm)"]
    table = pd.DataFrame(rows)[cols]
    # "Int64" (capital I) = integer column that may contain missing values (<NA>)
    table[["Ref. ID", "Pred. ID"]] = table[["Ref. ID", "Pred. ID"]].astype("Int64")
    return table


# Compute the per-lesion table for each algorithm
lesion_tables = {}
for name in PRED_NAMES:
    matches, missed, false_pos = matching[name]
    lesion_tables[name] = lesionwise_metrics(ref_inst, pred_inst[name][0],
                                             matches, missed, false_pos, spacing_zyx)

Let's look at the per-lesion results of each algorithm. Pay attention to the **lesion size** column: are small lesions segmented as well as large ones?

### 18.1 Per-lesion results: pred1

In [ ]:
# Cell 18.2: Per-Lesion Results of pred1
lesion_tables["pred1"]

### 18.2 Per-lesion results: pred2

In [ ]:
# Cell 18.3: Per-Lesion Results of pred2
lesion_tables["pred2"]

### 18.3 Per-lesion results: pred3

In [ ]:
# Cell 18.4: Per-Lesion Results of pred3
lesion_tables["pred3"]

---
## SECTION 19: Lesion-wise Averages vs. Global Metrics

To summarise each algorithm with a single number per metric, we **average over lesions**. There are two common variants, and we report both:

* **Matched lesions only** — "when a lesion is found, how well is it delineated?" (segmentation quality, ignores detection errors).
* **All lesions (TP + FN + FP)** — unmatched lesions count with Dice = Precision = Recall = 0, so detection errors are penalised.

HD95 is averaged over matched lesions in both cases (it is undefined for unmatched lesions).

Every lesion now has the **same weight**, regardless of its size — exactly the opposite of the global metrics, where each *voxel* has the same weight.

In [ ]:
# Cell 19.1: Global vs. Lesion-wise Summary Table
METRICS = ["Dice", "Precision", "Recall", "HD95 (mm)"]

summary = {}
for name in PRED_NAMES:
    t = lesion_tables[name]
    matched = t[t["Type"] == "TP"]
    summary[name] = {
        # ---- Global (after binarisation) ------------------------------------
        **{("Global", m): global_fixed.loc[name, m] for m in METRICS},
        # ---- Lesion-wise mean over matched lesions only ---------------------
        **{("Lesion-wise, matched only", m): matched[m].mean() for m in METRICS},
        # ---- Lesion-wise mean over all lesions (FN/FP penalised) ------------
        **{("Lesion-wise, all lesions", m): t[m].mean() for m in METRICS[:3]},
        ("Lesion-wise, all lesions", "HD95 (mm)"): matched["HD95 (mm)"].mean(),
    }

comparison = pd.DataFrame(summary).T
comparison.columns = pd.MultiIndex.from_tuples(comparison.columns)
comparison.index.name = "Algorithm"
comparison

Tables full of numbers are hard to compare — let's plot the Dice and HD95 side by side.

In [ ]:
# Cell 19.2: Bar Charts of Dice and HD95
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(PRED_NAMES))          # one group of bars per algorithm
variants = ["Global", "Lesion-wise, matched only", "Lesion-wise, all lesions"]
bar_width = 0.27

for ax, metric in zip(axes, ["Dice", "HD95 (mm)"]):
    for k, variant in enumerate(variants):
        values = comparison[(variant, metric)].values
        bars = ax.bar(x + (k - 1) * bar_width, values, bar_width, label=variant)
        ax.bar_label(bars, fmt="%.2f", fontsize=8)      # print value on top of each bar
    ax.set_xticks(x)
    ax.set_xticklabels(PRED_NAMES)
    ax.set_title(metric + ("  (higher is better)" if metric == "Dice" else "  (lower is better)"))
    ax.spines[["top", "right"]].set_visible(False)

axes[0].set_ylim(0, 1.05)
axes[0].legend(loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

Finally, a scatter plot of **per-lesion Dice vs. lesion size** shows *why* global and lesion-wise numbers differ: global Dice is dominated by the largest lesions on the right, while lesion-wise averages give every point the same weight.

In [ ]:
# Cell 19.3: Per-Lesion Dice vs. Lesion Size
fig, ax = plt.subplots(figsize=(7, 4.5))
markers = {"pred1": "o", "pred2": "s", "pred3": "^"}

for name in PRED_NAMES:
    t = lesion_tables[name]
    ref_lesions = t[t["Type"] != "FP"]        # one point per REFERENCE lesion (TP or FN)
    ax.scatter(ref_lesions["Ref. size (vox)"], ref_lesions["Dice"],
               marker=markers[name], s=60, alpha=0.7, label=name)

ax.set_xscale("log")                          # lesion sizes span several orders of magnitude
ax.set_xlabel("Reference lesion size (voxels, log scale)")
ax.set_ylabel("Per-lesion Dice (0 = missed)")
ax.set_ylim(-0.05, 1.05)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Per-lesion Dice vs. lesion size")
plt.tight_layout()
plt.show()

---
## SECTION 20: Discussion and Take-Home Messages

Look back at the three tables (Sections 14, 16 and 19) (naive global → binarised global → lesion-wise) and answer for yourself:

1. **Does the ranking of the three algorithms stay the same** across global and lesion-wise metrics? If it changes, which ranking would you trust for a clinical application where every lesion matters?
2. **How large is the gap between global and lesion-wise Dice?** A large gap usually means that the big lesion is segmented well, but small lesions are missed or poorly delineated.
3. **How many false-positive lesions** does each algorithm produce? Global precision hides this almost completely when the spurious blobs are small.
4. **What happened to algorithm 3** between the naive and the corrected evaluation? What would have happened if you had published the naive number?

---
# PART 3: Summary & Master Verification

---

## SECTION 22: Summary of the Complete Workflow

| Step | Key takeaway |
|---|---|
| DICOM series | Slice order comes from the header geometry (IPP projected on the normal $\mathbf{r}\times\mathbf{c}$), never from filenames |
| Geometry | The affine combines direction cosines (IOP), spacing (PixelSpacing + slice spacing) and the origin (IPP). DICOM/ITK use **LPS**, NIfTI uses **RAS** |
| Intensities | `RescaleSlope`/`RescaleIntercept` → HU; display with an appropriate window |
| Libraries | SimpleITK reproduces our manual reconstruction; NIfTI keeps only voxels + geometry |
| Global metrics | Dice, Precision, Recall, HD95 (in mm!) treat all lesion voxels as one object; large lesions dominate |
| Label pitfalls | Always inspect `np.unique` of every mask; `pred == 1` silently dropped label 2 of pred3 |
| Lesion-wise metrics | Connected components + Hungarian IoU matching → detection (TP/FN/FP lesions) and per-lesion quality |
| Reporting | Report global **and** lesion-wise metrics, together with detection counts and the matching rules (connectivity, IoU threshold) |